# Gemini version of 04a — cosine similarity to adult mean

Parallel to `04a_draw_generate_cossim_adults`, using the Gemini full/half-time drawing
embeddings from `03g_draw_generate_gemini_embedding` instead of CLIP's. No column-wise
z-scoring here, as gemini-embedding-2 model's embeddings are already L2-normalized.

Output: `data/preprocessed/gemini/draw_cossim_adults.csv`, same columns as the CLIP
version (`participant_id, age_months, target_category, draw_cossim_adults,
draw_half_cossim_adults`) so the two are directly comparable.

The half-time section is skipped gracefully if `draw_gemini_embeddings_half.csv`
doesn't exist yet (e.g. have not run it yet due to API quota)
but will eventually populate `draw_half_cossim_adults`.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
import ast

SEA_ANIMALS = ["crab", "octopus", "seahorse", "shark", "whale", "turtle"]
DIM = 768  # gemini-embedding-2 output dimension

FULL_EMBED_CSV = "../data/embeddings/gemini/draw_gemini_embeddings_full.csv"
HALF_EMBED_CSV = "../data/embeddings/gemini/draw_gemini_embeddings_half.csv"
OUT_CSV = "../data/preprocessed/gemini/draw_cossim_adults.csv"

FLAGS_CSV = "../data/tidy/drawings.csv"  # interference flags

# Interference trials are excluded throughout, so they enter neither the adult mean nor the output
_flags = pd.read_csv(FLAGS_CSV)
INTERFERENCE = set(zip(_flags.loc[_flags["draw_interference"] == 1, "participant_id"].str.strip(),
                       _flags.loc[_flags["draw_interference"] == 1, "target_category"].str.strip().str.lower()))

In [2]:
def parse_embedding(x):
    if isinstance(x, np.ndarray):
        return x
    if isinstance(x, list):
        return np.array(x, dtype=float)
    if pd.isna(x):
        return np.nan
    return np.array(ast.literal_eval(x), dtype=float)


def mean_embedding(emb_series):
    arr = np.vstack(emb_series.to_list())
    return arr.mean(axis=0)

## Full drawings

In [3]:
full_df = pd.read_csv(FULL_EMBED_CSV)
full_df["embedding"] = full_df["embedding"].apply(parse_embedding)
full_df = full_df[full_df["target_category"].isin(SEA_ANIMALS)].copy()
full_df = full_df[[k not in INTERFERENCE for k in zip(full_df["participant_id"], full_df["target_category"])]].copy()
print("Full drawings (sea animals):", full_df.shape)
print(full_df["is_adult"].value_counts())

Full drawings (sea animals): (610, 6)
is_adult
False    490
True     120
Name: count, dtype: int64


In [4]:
adult_means_full = (
    full_df[full_df["is_adult"]]
    .groupby("target_category")["embedding"]
    .apply(mean_embedding)
)
full_df["adult_mean"] = full_df["target_category"].map(adult_means_full)

mask = (
    full_df["adult_mean"].notna()
    & full_df["embedding"].apply(lambda x: isinstance(x, np.ndarray) and len(x) == DIM)
)
E = np.vstack(full_df.loc[mask, "embedding"].to_list()).astype(float)
A = np.vstack(full_df.loc[mask, "adult_mean"].to_list()).astype(float)

num = np.sum(E * A, axis=1)
den = np.linalg.norm(E, axis=1) * np.linalg.norm(A, axis=1)
full_df.loc[mask, "draw_cossim_adults"] = num / den

# Drop adults — they were used to build the mean, not the output
full_df = full_df[~full_df["is_adult"]].copy()
print(full_df["draw_cossim_adults"].describe())

count    490.000000
mean       0.885379
std        0.054789
min        0.689904
25%        0.841645
50%        0.890629
75%        0.932070
max        0.980652
Name: draw_cossim_adults, dtype: float64


## Half-time drawings

In [5]:
have_half = Path(HALF_EMBED_CSV).exists()
if have_half:
    half_df = pd.read_csv(HALF_EMBED_CSV)
    half_df["embedding"] = half_df["embedding"].apply(parse_embedding)
    half_df = half_df[half_df["target_category"].isin(SEA_ANIMALS)].copy()
    half_df = half_df[[k not in INTERFERENCE for k in zip(half_df["participant_id"], half_df["target_category"])]].copy()
    print("Half-time drawings (sea animals):", half_df.shape)
    print(half_df["is_adult"].value_counts())

    adult_means_half = (
        half_df[half_df["is_adult"]]
        .groupby("target_category")["embedding"]
        .apply(mean_embedding)
    )
    half_df["adult_mean"] = half_df["target_category"].map(adult_means_half)

    mask_h = (
        half_df["adult_mean"].notna()
        & half_df["embedding"].apply(lambda x: isinstance(x, np.ndarray) and len(x) == DIM)
    )
    Eh = np.vstack(half_df.loc[mask_h, "embedding"].to_list()).astype(float)
    Ah = np.vstack(half_df.loc[mask_h, "adult_mean"].to_list()).astype(float)

    numh = np.sum(Eh * Ah, axis=1)
    denh = np.linalg.norm(Eh, axis=1) * np.linalg.norm(Ah, axis=1)
    half_df.loc[mask_h, "draw_half_cossim_adults"] = numh / denh

    half_df = half_df[~half_df["is_adult"]].copy()
    print(half_df["draw_half_cossim_adults"].describe())
else:
    print(f"!! {HALF_EMBED_CSV} not found yet — half-time embedding is still pending "
          "(quota or not yet run). Producing full-drawing cossim only; re-run this "
          "notebook once it exists to backfill draw_half_cossim_adults.")
    half_df = None

!! ../data/embeddings/gemini/draw_gemini_embeddings_half.csv not found yet — half-time embedding is still pending (quota or not yet run). Producing full-drawing cossim only; re-run this notebook once it exists to backfill draw_half_cossim_adults.


In [6]:
if have_half:
    result = full_df[["participant_id", "age_months", "target_category", "draw_cossim_adults"]].merge(
        half_df[["participant_id", "target_category", "draw_half_cossim_adults"]],
        on=["participant_id", "target_category"],
        how="outer",
    )
else:
    result = full_df[["participant_id", "age_months", "target_category", "draw_cossim_adults"]].copy()
    result["draw_half_cossim_adults"] = np.nan

print(result.groupby("target_category")[["draw_cossim_adults", "draw_half_cossim_adults"]].describe().round(3))

Path(OUT_CSV).parent.mkdir(parents=True, exist_ok=True)
result.to_csv(OUT_CSV, index=False)
print(f"\nSaved {result.shape[0]} rows -> {OUT_CSV}")

                draw_cossim_adults                                            \
                             count   mean    std    min    25%    50%    75%   
target_category                                                                
crab                          81.0  0.884  0.054  0.762  0.849  0.880  0.936   
octopus                       82.0  0.883  0.056  0.754  0.836  0.899  0.927   
seahorse                      81.0  0.893  0.041  0.753  0.868  0.900  0.925   
shark                         83.0  0.880  0.065  0.728  0.823  0.881  0.939   
turtle                        81.0  0.894  0.052  0.765  0.854  0.901  0.936   
whale                         82.0  0.879  0.058  0.690  0.832  0.876  0.931   

                       draw_half_cossim_adults                               
                   max                   count mean std min 25% 50% 75% max  
target_category                                                              
crab             0.976                     0.